# Combined dataset audit

**AAI-590 | Gaurav: threat modeling, evaluation protocol and abstention policy**

This notebook combines the three existing audits, in their original order:

1. [Dataset understanding](#part-1)
2. [Source-to-database reconciliation](#part-2)
3. [Timing and metadata provenance](#part-3)

**Run:** select Python 3.11+ (your `rocm-jupyter` kernel or the project environment),
restart once, then **Run All**. Start from the project root or `notebooks/`.
Do not restart between parts. The extracted `data/radio` files must be present.
Notebook packages/setup instructions are in [README](../README.md).

The original notebooks are preserved. All 23 nonempty original code cells are
included unchanged; each part retains its environment setup so it can disclose
its own module identity. Saved outputs below come from this combined notebook's
execution, not copied original results. Raw-data access is read-only. The
optional ZIP check reads directory entries only; there are no network requests.

**Read the narrative chronologically:** original decision-gate text describes
what was unresolved at that audit stage. Later parts investigate those questions.
For current task/approval status use [the checklist](../planning/checklist.md),
[the evaluation protocol](../docs/evaluation_protocol.md),
[the threat model](../docs/threat_model.md) and
[the abstention policy](../docs/abstention_policy.md). Historical `pending` text
below does not revoke later approvals.

This is a combined dataset/provenance audit, not model training or policy
execution. Candidate matches and file/UE streams do not establish independent
captures, authenticated clocks or verified attack outcomes.


<a id="part-1"></a>

## Part 1: Dataset understanding

Source: [01_dataset_understanding.ipynb](01_dataset_understanding.ipynb). Historical audit narrative follows. Run continuously without restarting.

## 01 - Dataset understanding and provenance

**AAI-590 | Gaurav's threat modeling, evaluation, and abstention workstream**

This notebook is the research narrative for the read-only structural audit.
It calls `src/reliability/data_audit.py`; reusable parsing and validation stay
in that tested module. It does not train models, generate degradations,
reconstruct independent runs, reconcile individual records, or select features.

**How to use:** select the project's `.venv` Python interpreter, use Run All from the top of this combined notebook. Launch from the project root or `notebooks/`. The
project environment can be set up with `requirements-notebooks.txt`; an
existing Python 3.11+ notebook kernel can also run the audit without pytest. The extracted
`data/radio/` files must be present. No earlier notebook state or saved results
are required. All file access here is read-only; the notebook editor may save
cell outputs into this notebook. Outputs contain aggregate diagnostics, not
raw telemetry rows.

**Evidence rule:** counts describe these input files, not independent
experimental runs. The checklist remains the task-status authority. Kaggle
release, timestamp semantics, and source-to-summary lineage remain open.


### 1. Locate the project and record the environment

Resolve paths from the working directory instead of using a machine-specific
absolute data path. If discovery fails, launch the notebook from the project
root or its `notebooks/` directory. The project `.venv` is recommended for a consistent environment. Other
Python 3.11+ kernels with IPython can run this standard-library audit; missing
pytest/nbformat/nbclient metadata is reported without stopping execution. Dependency versions and the audit code hash identify
the execution context; they are not evidence that the dataset labels are valid.


In [1]:
from pathlib import Path
import hashlib
import html
import importlib.metadata
import json
import platform
import sys
from IPython.display import HTML, Markdown, display

candidates = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (p for p in candidates if (p / 'src/reliability/data_audit.py').is_file()
     and (p / 'planning/checklist.md').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Launch from the project root or notebooks directory.')

source_path = str(PROJECT_ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
from reliability.data_audit import build_inventory

DATA_ROOT = PROJECT_ROOT / 'data/radio'
if not (DATA_ROOT / 'Lower_Layer_Data.db').is_file():
    raise FileNotFoundError('Extract the radio subset into data/radio first.')

# Presentation only: analysis and data parsing remain in the Python module.
def show_table(headers, rows):
    def cell(value):
        return html.escape(str(value))
    head = ''.join(f'<th>{cell(h)}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{cell(v)}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

audit_code_sha256 = hashlib.sha256(
    (PROJECT_ROOT / 'src/reliability/data_audit.py').read_bytes()
).hexdigest()
print('Python:', platform.python_version())
print('Interpreter:', sys.executable)
print('SQLite:')
import sqlite3
print(sqlite3.sqlite_version)
print('Data path relative to project:', DATA_ROOT.relative_to(PROJECT_ROOT).as_posix())
print('Audit code SHA-256:', audit_code_sha256)
def optional_package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'Not installed (optional for interactive audit)'

expected_interpreter = PROJECT_ROOT / '.venv/Scripts/python.exe'
if Path(sys.executable).resolve() != expected_interpreter.resolve():
    print('Using a different interpreter from the project .venv; versions are recorded below.')
    print('The audit needs Python 3.11+; optional test/execution tools need not be installed here.')

show_table(['Package', 'Version'], [
    (name, optional_package_version(name))
    for name in ('pytest', 'nbformat', 'nbclient', 'ipykernel')
])


Python: 3.11.14
Interpreter: c:\Users\saiga\miniforge3\envs\rocm-jupyter\python.exe
SQLite:
3.51.0
Data path relative to project: data/radio
Audit code SHA-256: 0fbf91bcb3f4586d257c9792105e4d5d59b4e564b21d99bf43e98d6d59c32eab
Using a different interpreter from the project .venv; versions are recorded below.
The audit needs Python 3.11+; optional test/execution tools need not be installed here.


Package,Version
pytest,Not installed (optional for interactive audit)
nbformat,5.10.4
nbclient,0.10.2
ipykernel,7.1.0


### 2. Recompute the inventory from the current input bytes

Do not silently trust a cached report. The audit verifies every selected file
against `extraction_manifest.json` when available, opens SQLite read-only,
parses source JSONL, and verifies input hashes again after inspection. A failed
integrity check raises an error and stops execution. No random sampling is used.

The report hash below identifies canonical UTF-8 JSON with LF line endings.
Windows CLI artifacts may use CRLF; compare parsed content, not platform-specific
file bytes. The separate artifact hash identifies the original file. This
notebook does not write it to disk. The command in the project README can
create a separate JSON artifact when needed.


In [2]:
report = build_inventory(DATA_ROOT)
report_bytes = (json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + chr(10)).encode('utf-8')
report_sha256 = hashlib.sha256(report_bytes).hexdigest()
print('Extraction manifest:', report['extraction_manifest_check'])
print('Input hashes unchanged:', report['input_hashes_unchanged'])
print('Input files:', len(report['input_files']))
print('Source text files:', len(report['sources']))
print('Canonical report SHA-256:', report_sha256)

previous_path = PROJECT_ROOT / 'results/radio_inventory_2026-11-13.json'
if previous_path.exists():
    previous_bytes = previous_path.read_bytes()
    previous_report = json.loads(previous_bytes)
    print('Prior artifact SHA-256:', hashlib.sha256(previous_bytes).hexdigest())
    print('Matches prior local inventory (content):', previous_report == report)
    if previous_report != report:
        print('Investigate code/input changes before comparing findings; the prior report is not used below.')
else:
    print('No prior report found; every table below uses this fresh inventory.')

Extraction manifest: all selected file sizes and SHA-256 verified
Input hashes unchanged: True
Input files: 50
Source text files: 49
Canonical report SHA-256: c2cb31d7eff08c9fd6cc357b48fb00f1bfee8ba58f95e70f58723236a1e56fd5
Prior artifact SHA-256: 83c90d85105cb8f2a5243b1d43dfb963182aeb9b3eb10afe7ae0cfd57f2523a9
Matches prior local inventory (content): True


### 3. Database schema, missingness, and literal labels

Keep schema facts separate from modeling decisions. The non-label columns
include timestamps and identifiers, so their count is not the number of
eligible predictors. A single distinct value is a descriptive diagnostic;
future learned feature selection and preprocessing must use training data only.
SQL NULL counts do not detect sentinel values, empty strings, or invalid units.


In [3]:
show_table(['Table', 'Rows', 'Columns'], [
    (name, table['rows'], len(table['columns']))
    for name, table in report['database']['tables'].items()
])
if 'lower_layer_data' not in report['database']['tables']:
    raise ValueError('Expected lower_layer_data table is absent; inspect the schema before continuing.')
table = report['database']['tables']['lower_layer_data']
show_table(['Column', 'Declared type', 'SQL NULLs', 'Distinct non-NULL'], [
    (c['name'], c['declared_type'], c['null_count'], c['distinct_nonnull'])
    for c in table['columns']
])
for label, counts in table['labels'].items():
    display(Markdown(f'**Literal values: `{label}`**'))
    show_table(['Value', 'Rows'], [(item['value'], item['rows']) for item in counts])


Table,Rows,Columns
lower_layer_data,45244,25


Column,Declared type,SQL NULLs,Distinct non-NULL
dlBytes,INTEGER,0,44737
dlMcs,INTEGER,0,23
dlBler,REAL,0,28623
ulBytes,INTEGER,0,43888
ulMcs,INTEGER,0,21
ulBler,REAL,0,30258
ri,INTEGER,0,1
phr,INTEGER,0,32
pcmax,INTEGER,0,6
rsrq,REAL,0,22


**Literal values: `traffic_type`**

Value,Rows
0,10664
1,34580


**Literal values: `attack_category`**

Value,Rows
Benign,10664
BruteForce,3057
DDoS,15084
DoS,4721
Probe,8445
Web Attacks,3273


**Literal values: `attack_subcategory`**

Value,Rows
Benign,10664
BruteForce_FTP,1391
BruteForce_SSH,1666
DDoS_ACK,1934
DDoS_ICMP,3294
DDoS_SYN,3568
DDoS_Slowloris,1830
DDoS_UDP,4458
DoS_ACK,648
DoS_ICMP,1037


### 4. Compare source counts with database counts

Sources have folder-derived categories, not validated per-record labels.
For display only, the aliases `DDOS -> DDoS` and `Web -> Web Attacks` align
folder and database category names. They do not modify input labels or define
a modeling interface. Unknown categories are retained in the comparison.

A matching count is **not** proof of matching records. A mismatch does not
identify the cause: filtering, repeated data, relabeling, or timestamp
population must be investigated using record-level evidence and processing
provenance. No row is discarded by this notebook.


In [4]:
aliases_for_display = {'DDOS': 'DDoS', 'Web': 'Web Attacks'}
source_totals = {}
for source in report['sources']:
    folder = source['category_from_folder']
    category = aliases_for_display.get(folder, folder)
    stats = source_totals.setdefault(category, {'files': 0, 'objects': 0, 'timestamped': 0})
    stats['files'] += 1
    stats['objects'] += source['valid_object_records']
    stats['timestamped'] += source['timestamp']['finite_numeric']

database_counts = {item['value']: item['rows'] for item in table['labels'].get('attack_category', [])}
category_rows = []
for category in sorted(set(source_totals) | set(database_counts), key=str):
    stats = source_totals.get(category, {'files': 0, 'objects': 0, 'timestamped': 0})
    database_rows = database_counts.get(category, 0)
    category_rows.append((category, stats['files'], stats['objects'], stats['timestamped'],
                          database_rows, stats['objects'] - database_rows))
show_table(['Category (display alias)', 'Source files', 'Source objects',
            'Timestamped objects', 'Database rows', 'Source minus DB'], category_rows)

source_records = sum(s['valid_object_records'] for s in report['sources'])
timestamped_records = sum(s['timestamp']['finite_numeric'] for s in report['sources'])
print('Source objects:', source_records)
print('Database rows:', table['rows'])
print('Source objects with finite numeric timestamps:', timestamped_records)
print('Count difference (source minus database):', source_records - table['rows'])


Category (display alias),Source files,Source objects,Timestamped objects,Database rows,Source minus DB
Benign,12,11598,10664,10664,934
BruteForce,4,3057,3057,3057,0
DDoS,9,15084,15084,15084,0
DoS,10,8187,3929,4721,3466
Probe,7,9014,8445,8445,569
Web Attacks,7,3273,3273,3273,0


Source objects: 50213
Database rows: 45244
Source objects with finite numeric timestamps: 44452
Count difference (source minus database): 4969


### 5. Identify source files with insufficient timestamp evidence

A missing field and a JSON null are different conditions. This table retains
that distinction. An invalid non-null timestamp is also reported explicitly.
The number of files is not the number of independent runs. Files named `test`
remain visible; their names alone do not justify exclusion.

Finite numeric timestamps do not establish clock units, timezone, ordering,
UE identity, freshness, or one-second reporting. We do not interpret gaps or
convert cumulative bytes to rates here.


In [5]:
problem_sources = [s for s in report['sources']
                   if s['timestamp']['finite_numeric'] != s['valid_object_records']]
show_table(['Source path', 'Objects', 'Timestamp absent', 'Timestamp null',
            'Invalid non-null', 'Finite numeric'], [
    (s['path'], s['valid_object_records'],
     s['fields'].get('timestamp', {}).get('absent', s['valid_object_records']),
     s['fields'].get('timestamp', {}).get('null', 0),
     s['timestamp']['invalid_nonnull'], s['timestamp']['finite_numeric'])
    for s in problem_sources
])
print('Malformed/non-object JSONL lines:', sum(len(s['invalid_records']) for s in report['sources']))
print('Database timestamp summary (raw units):')
print(json.dumps(table['timestamp'], indent=2))


Source path,Objects,Timestamp absent,Timestamp null,Invalid non-null,Finite numeric
Benign/Benign/Lower_Layer/benign_10.txt,900,900,0,0,0
Benign/Benign/Lower_Layer/benign_test.txt,34,34,0,0,0
DoS/DoS/Lower_Layer/icmp_hping3.txt,1125,1125,0,0,0
DoS/DoS/Lower_Layer/slowloris.txt,1007,1007,0,0,0
DoS/DoS/Lower_Layer/tcp-ack_hping3.txt,648,648,0,0,0
DoS/DoS/Lower_Layer/tcp-syn_hping3.txt,768,768,0,0,0
DoS/DoS/Lower_Layer/tcp_test.txt,124,124,0,0,0
DoS/DoS/Lower_Layer/udp_hping3.txt,586,586,0,0,0
Probe/Probe/Lower_Layer/portscan_udp.txt,1142,569,0,0,573


Malformed/non-object JSONL lines: 0
Database timestamp summary (raw units):
{
  "present_nonnull": 45244,
  "finite_numeric": 45244,
  "invalid_nonnull": 0,
  "raw_min": 1746799506701.0,
  "raw_max": 1751375357835.0,
  "distinct_numeric": 31550,
  "units_and_timezone": "unverified"
}


### 6. Inspect source schema variation

Source records can use different metadata and PMI representations. The table
below counts absent/null fields across valid objects without transforming
any representation. A column present somewhere in a file can still be absent
from some records in that file. This matters when defining an eventual
source-to-database comparison key.

The present implementation is a structural audit, not a physical-value check,
duplicate-key detector, or rowwise label-consistency check. Parsing success
alone does not establish valid measurements.


In [6]:
fields_to_review = ('timestamp', 'ue_id', 'cellid', 'in_sync', 'rnti', 'pmi', 'pmi_0', 'pmi_1')
field_rows = []
for key in fields_to_review:
    absent = nulls = files_with_field = 0
    for source in report['sources']:
        field = source['fields'].get(key)
        if field is None:
            absent += source['valid_object_records']
        else:
            files_with_field += 1
            absent += field['absent']
            nulls += field['null']
    field_rows.append((key, files_with_field, absent, nulls))
show_table(['Source field', 'Files containing field', 'Objects with field absent',
            'Objects with field null'], field_rows)


Source field,Files containing field,Objects with field absent,Objects with field null
timestamp,41,5761,0
ue_id,41,5761,0
cellid,41,5761,0
in_sync,41,5761,0
rnti,41,5761,0
pmi,9,41906,0
pmi_0,40,8307,0
pmi_1,40,8307,0


### 7. Interpret the evidence before choosing a grouping rule

- The database retains no explicit source filename or run ID in its observed
  schema. Its timestamps alone are not a validated run manifest.
- Differences between source and database counts remain unexplained. Neither
  source file boundaries nor timestamp gaps establish independent captures.
- Four constant database columns and missing `rnti` values are descriptive
  observations, not approval of a predictor list or imputation rule.
- Counting rows or source files cannot substantiate a very low false-positive
  rate. That requires independent benign capture support and an approved
  evaluation/uncertainty protocol.
- Recording-day confounding and one-second sampling remain hypotheses until
  time semantics, lineage, and class/date support are validated.
- This audit does not establish malicious telemetry loss. Missing observations
  are a data-quality issue whose causes remain unknown.

Global inventory statistics are for integrity and feasibility. Do not use them
to fit transformations or select features using a future held-out partition.


### 8. Next investigation: source-to-database reconciliation (not implemented)

The next manageable task is a diagnostic reconciliation, starting with DoS
and the mixed-schema Probe file. Before implementing it:

1. Obtain the Kaggle release and any author processing scripts. Distinguish
   content version 2 from Kaggle's platform revision number.
2. Compare actual source/database field representations; document any
   necessary normalization separately from original values. Never fabricate
   missing timestamps or coerce an ambiguous match into a unique one.
3. Propose exact comparison fields and an explicit treatment of multiplicity.
   Report unique, ambiguous, unmatched, and repeated matches in both directions.
   Matching telemetry tuples does not itself validate independent runs.
4. Explain discrepancies before proposing source eligibility, run boundaries,
   evaluation splits, or counter-to-rate transformations for Gaurav's review.

**Team boundaries:** Nitendra owns model training/calibration; Marco owns
telemetry degradation/playback. This notebook supplies evidence to Gaurav's
threat model, evaluation protocol, and abstention requirements.

**Review prompts:** What provenance explains the DoS count difference? Which
source records lack clocks? Can source identities be recovered without
choosing arbitrary matches? Which questions remain unidentifiable?

See `docs/data_audit_findings.md` for the initial written findings,
`planning/checklist.md` for task status, and `docs/decisions.md` for choices.


In [7]:
assert report['input_hashes_unchanged']
print('Structural inventory complete; input bytes unchanged during the audit.')
print('Source-to-database reconciliation: pending.')
print('Independent run validation: pending.')
print('No models, counter transformations, row exclusions, or evaluation splits created.')


Structural inventory complete; input bytes unchanged during the audit.
Source-to-database reconciliation: pending.
Independent run validation: pending.
No models, counter transformations, row exclusions, or evaluation splits created.


<a id="part-2"></a>

## Part 2: Source-to-database reconciliation

Source: [02_source_database_reconciliation.ipynb](02_source_database_reconciliation.ipynb). Historical audit narrative follows. Run continuously without restarting.

## 02 - Source-to-database reconciliation

**AAI-590 | Gaurav's dataset provenance diagnostic**

This part runs after Part 1 in the same Python 3.11+ kernel. This independent notebook calls the tested reconciliation module
on all six current classes, with DoS and Probe as the main investigation.
No cached result is required. It does not alter raw files or assign run IDs.

The preceding notebook established count differences. Here we ask which
observations agree on explicitly selected fields, preserving every occurrence.
Exact candidate agreement is not authoritative capture provenance.

### 1. Resolve paths and disclose the execution environment

The project `.venv` is recommended; the existing rocm-jupyter kernel can also
run this standard-library diagnostic. Missing optional test/execution tools
are reported without preventing the audit. Setup contains presentation code;
matching logic stays in `src/reliability/reconciliation.py`.

In [8]:
from pathlib import Path
import hashlib
import html
import importlib.metadata
import json
import platform
import sys
from IPython.display import HTML, Markdown, display

candidates = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (p for p in candidates if (p / 'src/reliability/data_audit.py').is_file()
     and (p / 'planning/checklist.md').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Launch from the project root or notebooks directory.')

source_path = str(PROJECT_ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
from reliability.reconciliation import build_reconciliation, PROFILES, STATUSES

DATA_ROOT = PROJECT_ROOT / 'data/radio'
if not (DATA_ROOT / 'Lower_Layer_Data.db').is_file():
    raise FileNotFoundError('Extract the radio subset into data/radio first.')

# Presentation only: analysis and data parsing remain in the Python module.
def show_table(headers, rows):
    def cell(value):
        return html.escape(str(value))
    head = ''.join(f'<th>{cell(h)}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{cell(v)}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

reconciliation_code_sha256 = hashlib.sha256(
    (PROJECT_ROOT / 'src/reliability/reconciliation.py').read_bytes()
).hexdigest()
print('Python:', platform.python_version())
print('Interpreter:', sys.executable)
print('SQLite:')
import sqlite3
print(sqlite3.sqlite_version)
print('Data path relative to project:', DATA_ROOT.relative_to(PROJECT_ROOT).as_posix())
print('Reconciliation code SHA-256:', reconciliation_code_sha256)
def optional_package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'Not installed (optional for interactive audit)'

expected_interpreter = PROJECT_ROOT / '.venv/Scripts/python.exe'
if Path(sys.executable).resolve() != expected_interpreter.resolve():
    print('Using a different interpreter from the project .venv; versions are recorded below.')
    print('The audit needs Python 3.11+; optional test/execution tools need not be installed here.')

show_table(['Package', 'Version'], [
    (name, optional_package_version(name))
    for name in ('pytest', 'nbformat', 'nbclient', 'ipykernel')
])


Python: 3.11.14
Interpreter: c:\Users\saiga\miniforge3\envs\rocm-jupyter\python.exe
SQLite:
3.51.0
Data path relative to project: data/radio
Reconciliation code SHA-256: 45a8de2d300ab6c579bed20c30c6873f4720d32df066f7f9dad9fc61ca32fea6
Using a different interpreter from the project .venv; versions are recorded below.
The audit needs Python 3.11+; optional test/execution tools need not be installed here.


Package,Version
pytest,Not installed (optional for interactive audit)
nbformat,5.10.4
nbclient,0.10.2
ipykernel,7.1.0


### 2. State comparison keys before interpreting results

Two diagnostic projections are fixed in the module. Neither includes labels,
so a different label cannot hide a candidate with identical compared values.
The radio key has 16 numeric fields. The stronger key adds timestamp, UE ID,
cell ID, sync state and literal RNTI. PMI is excluded rather than silently
converted between `pmi` and `pmi_0`/`pmi_1` representations.

Numbers are compared exactly as rational values: integer 1 equals float 1.0;
no rounding or tolerance is introduced. Numeric strings, booleans, nonfinite
values and missing/null fields cannot supply a numeric key. No missing
metadata is filled. RNTI strings are compared literally and case-sensitively.
These projections are diagnostic choices, not model features or split rules.

In [9]:
for name, fields in PROFILES.items():
    print(name, ':', len(fields), 'fields')
    print(', '.join(fields))

radio_exact : 16 fields
dlBytes, dlMcs, dlBler, ulBytes, ulMcs, ulBler, ri, phr, pcmax, rsrq, sinr, rsrp, rssi, cqi, pucchSnr, puschSnr
radio_metadata_exact : 21 fields
dlBytes, dlMcs, dlBler, ulBytes, ulMcs, ulBler, ri, phr, pcmax, rsrq, sinr, rsrp, rssi, cqi, pucchSnr, puschSnr, timestamp, ue_id, cellid, in_sync, rnti


### 3. Recompute with integrity checks

The module first verifies the extraction manifest through the inventory audit,
reads source records with original relative paths and physical line numbers,
and opens SQLite read-only. Database rowids are local row references, not run
IDs. Every occurrence is retained, and all 50 file hashes are checked again
before returning. The result stays in memory; the CLI can save a detailed
JSON artifact when required.

In [10]:
reconciliation = build_reconciliation(DATA_ROOT)
canonical_bytes = (json.dumps(reconciliation, indent=2, sort_keys=True, allow_nan=False) + chr(10)).encode('utf-8')
print('Source occurrences:', len(reconciliation['source_records']))
print('Database rows:', len(reconciliation['database_records']))
print('Extraction manifest:', reconciliation['extraction_manifest_check'])
print('Input hashes unchanged:', reconciliation['input_hashes_unchanged'])
print('Canonical reconciliation SHA-256:', hashlib.sha256(canonical_bytes).hexdigest())
prior_path = PROJECT_ROOT / 'results/radio_reconciliation_2026-11-13.json'
if prior_path.exists():
    print('Matches prior CLI reconciliation (content):', json.loads(prior_path.read_bytes()) == reconciliation)
else:
    print('No saved CLI report present; all following tables use fresh results.')
source_refs = {s['id']: s for s in reconciliation['source_records']}
database_refs = {d['rowid']: d for d in reconciliation['database_records']}

Source occurrences: 50213
Database rows: 45244
Extraction manifest: all selected file sizes and SHA-256 verified
Input hashes unchanged: True
Canonical reconciliation SHA-256: dab06dc5e92f5ed526a07fc4c7e7e89ddfa5575ae3ec523ad662ea7be8962c25
Matches prior CLI reconciliation (content): True


### 4. Preserve candidate multiplicity

For each usable comparison key, consider **all** occurrences on both sides:

- `unique_candidate_pair`: one source occurrence and one database occurrence.
- `ambiguous_candidate_group`: candidates exist on both sides, but at least
  one side has repeated occurrences. No pairing is selected or consumed.
- `no_candidate`: the key is usable, but no occurrence exists on the other side.
- `unusable_key`: a required field is missing/null or invalid; this is not
  evidence that the radio record is absent from the database.

These categories partition each side separately. Stronger-key ineligibility
must not be counted as unmatched or silently excluded from reporting.

In [11]:
summary_rows = []
for name, profile in reconciliation['profiles'].items():
    for side in ('source_counts', 'database_counts'):
        counts = profile[side]
        summary_rows.append([name, side, *[counts[s] for s in STATUSES], sum(counts.values())])
show_table(['Profile', 'Side', *STATUSES, 'Total'], summary_rows)
for name, profile in reconciliation['profiles'].items():
    print(name, 'unique-pair folder/category disagreements:',
          len(profile['unique_pair_folder_label_disagreements']))

Profile,Side,unique_candidate_pair,ambiguous_candidate_group,no_candidate,unusable_key,Total
radio_exact,source_counts,44819,425,4969,0,50213
radio_exact,database_counts,44819,425,0,0,45244
radio_metadata_exact,source_counts,43704,0,748,5761,50213
radio_metadata_exact,database_counts,43704,0,0,1540,45244


radio_exact unique-pair folder/category disagreements: 0
radio_metadata_exact unique-pair folder/category disagreements: 0


### 5. Inspect database category support and source files

Folder aliases DDOS/DDoS and Web/Web Attacks are diagnostic label comparisons
only. They do not participate in key construction. Zero unique-pair label
disagreements would support consistency for those compared observations,
not validate every label, subtype, binary flag or collection condition.

In [12]:
for name, profile in reconciliation['profiles'].items():
    display(Markdown(f'**Database categories: {name}**'))
    show_table(['Category', *STATUSES], [
        [c['category'], *[c[s] for s in STATUSES]] for c in profile['database_categories']
    ])
    display(Markdown(f'**DoS and Probe source files: {name}**'))
    show_table(['Source path', *STATUSES], [
        [f['path'], *[f[s] for s in STATUSES]] for f in profile['source_files']
        if f['path'].startswith(('DoS/', 'Probe/'))
    ])

**Database categories: radio_exact**

Category,unique_candidate_pair,ambiguous_candidate_group,no_candidate,unusable_key
Benign,10239,425,0,0
BruteForce,3057,0,0,0
DDoS,15084,0,0,0
DoS,4721,0,0,0
Probe,8445,0,0,0
Web Attacks,3273,0,0,0


**DoS and Probe source files: radio_exact**

Source path,unique_candidate_pair,ambiguous_candidate_group,no_candidate,unusable_key
DoS/DoS/Lower_Layer/icmp_hping3.txt,0,0,1125,0
DoS/DoS/Lower_Layer/icmp_nping.txt,1037,0,0,0
DoS/DoS/Lower_Layer/slowloris.txt,0,0,1007,0
DoS/DoS/Lower_Layer/slowloris2.txt,1104,0,0,0
DoS/DoS/Lower_Layer/tcp-ack_hping3.txt,648,0,0,0
DoS/DoS/Lower_Layer/tcp-syn_hping3.txt,768,0,0,0
DoS/DoS/Lower_Layer/tcp_npig.txt,0,0,748,0
DoS/DoS/Lower_Layer/tcp_test.txt,124,0,0,0
DoS/DoS/Lower_Layer/udp_hping3.txt,0,0,586,0
DoS/DoS/Lower_Layer/udp_nping.txt,1040,0,0,0


**Database categories: radio_metadata_exact**

Category,unique_candidate_pair,ambiguous_candidate_group,no_candidate,unusable_key
Benign,10664,0,0,0
BruteForce,3057,0,0,0
DDoS,15084,0,0,0
DoS,3181,0,0,1540
Probe,8445,0,0,0
Web Attacks,3273,0,0,0


**DoS and Probe source files: radio_metadata_exact**

Source path,unique_candidate_pair,ambiguous_candidate_group,no_candidate,unusable_key
DoS/DoS/Lower_Layer/icmp_hping3.txt,0,0,0,1125
DoS/DoS/Lower_Layer/icmp_nping.txt,1037,0,0,0
DoS/DoS/Lower_Layer/slowloris.txt,0,0,0,1007
DoS/DoS/Lower_Layer/slowloris2.txt,1104,0,0,0
DoS/DoS/Lower_Layer/tcp-ack_hping3.txt,0,0,0,648
DoS/DoS/Lower_Layer/tcp-syn_hping3.txt,0,0,0,768
DoS/DoS/Lower_Layer/tcp_npig.txt,0,0,748,0
DoS/DoS/Lower_Layer/tcp_test.txt,0,0,0,124
DoS/DoS/Lower_Layer/udp_hping3.txt,0,0,0,586
DoS/DoS/Lower_Layer/udp_nping.txt,1040,0,0,0


### 6. Investigate radio candidates whose source metadata is unavailable

This cross-profile table identifies unique radio candidates for which the
source cannot supply the stronger metadata key. The database's populated
metadata cannot be validated from those source records. Do not infer that it
was generated, copied or measured by a particular process without evidence.
This is the main limit on timestamp-based grouping and counter-rate conversion.

In [13]:
from collections import Counter
radio = reconciliation['profiles']['radio_exact']
strong = reconciliation['profiles']['radio_metadata_exact']
unusable_source_ids = {item['id'] for item in strong['unusable_sources']}
metadata_gaps = Counter()
for group in radio['groups']:
    if group['status'] == 'unique_candidate_pair' and group['source_ids'][0] in unusable_source_ids:
        source = source_refs[group['source_ids'][0]]
        database = database_refs[group['database_rowids'][0]]
        metadata_gaps[(source['path'], database['attack_category'], database['attack_subcategory'])] += 1
show_table(['Source file', 'Database category', 'Database subtype', 'Unique radio candidates'],
           [(*key, count) for key, count in sorted(metadata_gaps.items())])
ambiguous_groups = [g for g in radio['groups'] if g['status'] == 'ambiguous_candidate_group']
print('Repeated radio-key groups:', len(ambiguous_groups))
print('Source occurrences in these groups:', sum(len(g['source_ids']) for g in ambiguous_groups))
print('Database occurrences in these groups:', sum(len(g['database_rowids']) for g in ambiguous_groups))

Source file,Database category,Database subtype,Unique radio candidates
DoS/DoS/Lower_Layer/tcp-ack_hping3.txt,DoS,DoS_ACK,648
DoS/DoS/Lower_Layer/tcp-syn_hping3.txt,DoS,DoS_SYN,768
DoS/DoS/Lower_Layer/tcp_test.txt,DoS,DoS_SYN,124


Repeated radio-key groups: 61
Source occurrences in these groups: 425
Database occurrences in these groups: 425


### 7. Interpret the count mismatch with occurrence-level evidence

The interpretation must follow the live tables above:

- Keys may uniquely identify candidate source occurrences within these files;
  they do not establish that captures are independent or labels are correct.
- Radio-only agreement for metadata-free sources cannot validate database
  timestamps. Keep their temporal provenance unresolved.
- Unmatched source records remain in the report with path/line references.
  They are not automatically ineligible for research or safe to discard.
- Repeated radio-value groups can represent different observations with the
  same selected values. They are not proof of duplicate captures or staleness.
- No tolerance search, PMI conversion, row exclusion, timestamp imputation,
  train/test split or model training has been performed.

The detailed JSON report maps groups to source IDs and database rowids; the
reference tables map those IDs to original source path/line or database labels
and observed metadata. Raw metric vectors are not copied into notebook outputs.


### 8. Next decision gate

Before run validation, investigate the provenance of metadata for DoS rows
whose radio candidates lack source metadata. Obtain author processing code
or a capture manifest where possible. Then inspect within-source/per-UE time
ordering, overlaps and gaps only for records with supportable clock provenance.
Any fallback grouping or exclusion rule needs Gaurav's review and limitations.

No run count or date/class confounding conclusion is established by this step.
The preliminary 52 reconstructed versus reported 42 paired runs remains open.
Nitendra retains model/calibration ownership; Marco retains degradation/playback.
See `docs/reconciliation_findings.md` and the project checklist for evidence
and current status.

In [14]:
assert reconciliation['input_hashes_unchanged']
for name, profile in reconciliation['profiles'].items():
    assert sum(profile['source_counts'].values()) == len(source_refs)
    assert sum(profile['database_counts'].values()) == len(database_refs)
print('All source and database occurrences accounted for; inputs unchanged.')
print('Run IDs, timestamp provenance, and evaluation grouping remain unresolved.')

All source and database occurrences accounted for; inputs unchanged.
Run IDs, timestamp provenance, and evaluation grouping remain unresolved.


<a id="part-3"></a>

## Part 3: Timing and metadata provenance

Source: [03_timing_and_metadata_provenance.ipynb](03_timing_and_metadata_provenance.ipynb). Historical audit narrative follows. Run continuously without restarting.

## 03 - Timing diagnostics and metadata provenance

**AAI-590 | Gaurav's evaluation/provenance workstream**

This part runs after Part 2 in the same Python 3.11+ kernel.
This notebook calls `src/reliability/timing_audit.py`, which recomputes the
candidate reconciliation and checks input hashes. Prior result JSON files are
optional. The archive inventory is optional; it reads only ZIP directory entries.
There are no network requests in these cells.

**Scope:** describe raw timestamps, file/UE identity groups, overlap and
metadata coincidences. Do not assign independent runs, convert raw units to
calendar dates or seconds, fill clocks, select records for modeling, compute
counter rates or create train/test splits. Notebooks 01 and 02 are preserved.

### 1. Record paths, interpreter and audit code

The project `.venv` is recommended; an existing rocm-jupyter kernel can run
the standard-library diagnostic. Optional tools are disclosed without being
required for the interactive audit.

In [15]:
from pathlib import Path
import hashlib
import html
import importlib.metadata
import json
import platform
import sys
from IPython.display import HTML, Markdown, display

candidates = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (p for p in candidates if (p / 'src/reliability/data_audit.py').is_file()
     and (p / 'planning/checklist.md').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Launch from the project root or notebooks directory.')

source_path = str(PROJECT_ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
from reliability.timing_audit import build_timing_audit, STREAM_FIELDS, COINCIDENCE_FIELDS

DATA_ROOT = PROJECT_ROOT / 'data/radio'
if not (DATA_ROOT / 'Lower_Layer_Data.db').is_file():
    raise FileNotFoundError('Extract the radio subset into data/radio first.')

# Presentation only: analysis and data parsing remain in the Python module.
def show_table(headers, rows):
    def cell(value):
        return html.escape(str(value))
    head = ''.join(f'<th>{cell(h)}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{cell(v)}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

timing_code_sha256 = hashlib.sha256(
    (PROJECT_ROOT / 'src/reliability/timing_audit.py').read_bytes()
).hexdigest()
print('Python:', platform.python_version())
print('Interpreter:', sys.executable)
print('SQLite:')
import sqlite3
print(sqlite3.sqlite_version)
print('Data path relative to project:', DATA_ROOT.relative_to(PROJECT_ROOT).as_posix())
print('Timing code SHA-256:', timing_code_sha256)
def optional_package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'Not installed (optional for interactive audit)'

expected_interpreter = PROJECT_ROOT / '.venv/Scripts/python.exe'
if Path(sys.executable).resolve() != expected_interpreter.resolve():
    print('Using a different interpreter from the project .venv; versions are recorded below.')
    print('The audit needs Python 3.11+; optional test/execution tools need not be installed here.')

show_table(['Package', 'Version'], [
    (name, optional_package_version(name))
    for name in ('pytest', 'nbformat', 'nbclient', 'ipykernel')
])


Python: 3.11.14
Interpreter: c:\Users\saiga\miniforge3\envs\rocm-jupyter\python.exe
SQLite:
3.51.0
Data path relative to project: data/radio
Timing code SHA-256: 17c13af7fc7f68871aaf73274292c55b7ad562be0c679bb5830b419eff28f5e5
Using a different interpreter from the project .venv; versions are recorded below.
The audit needs Python 3.11+; optional test/execution tools need not be installed here.


Package,Version
pytest,Not installed (optional for interactive audit)
nbformat,5.10.4
nbclient,0.10.2
ipykernel,7.1.0


### 2. Recompute diagnostics with input integrity checks

Reconciliation provides exact candidate references; every source observation
is then inspected in physical file order. All 50 extracted input file hashes
are checked again at the end. No radio data is changed. The report below stays
in memory; the CLI documented in README saves a separate detailed JSON file.

The ZIP filename and size are provenance hints, not a verified Kaggle release.
We do not fully hash or decompress the archive here.

In [16]:
archive_path = PROJECT_ROOT / 'data/archive_2.zip'
timing = build_timing_audit(DATA_ROOT, archive_path if archive_path.exists() else None)
canonical_bytes = (json.dumps(timing, indent=2, sort_keys=True, allow_nan=False) + chr(10)).encode('utf-8')
print('Timing report SHA-256:', hashlib.sha256(canonical_bytes).hexdigest())
print('Extraction manifest:', timing['extraction_manifest_check'])
print('Input hashes unchanged:', timing['input_hashes_unchanged'])
print('Time units/timezone:', timing['units_and_timezone'])
print('Stream grouping fields (within each file):', ', '.join(STREAM_FIELDS))
print('Metadata coincidence fields:', ', '.join(COINCIDENCE_FIELDS))
prior_path = PROJECT_ROOT / 'results/timing_audit_2026-11-15.json'
if prior_path.exists():
    print('Matches prior CLI report (content):', json.loads(prior_path.read_bytes()) == timing)
    print('A different optional archive availability changes that inventory section.')

Timing report SHA-256: 1327941727dc090df7404c6558d5fa0a988fbffb75cd7dedb81da2cec6508cdf
Extraction manifest: all selected file sizes and SHA-256 verified
Input hashes unchanged: True
Time units/timezone: unverified; all differences in raw units
Stream grouping fields (within each file): cellid, ue_id, rnti
Metadata coincidence fields: timestamp, ue_id, cellid, in_sync
Matches prior CLI report (content): True
A different optional archive availability changes that inventory section.


### 3. Account for all source records

A diagnostic stream is one source path plus the observed tuple
`(cellid, ue_id, rnti)`. It is not a run ID or an independent capture. Records
without usable identity remain explicitly listed in the detailed report.
A changed identifier may split an otherwise related capture; a reused
identifier may span several captures. No population restriction is made.

In [17]:
from collections import Counter
categories = sorted({f['path'].split('/')[0] for f in timing['source_files']})
rows=[]
for category in categories:
    files=[f for f in timing['source_files'] if f['path'].split('/')[0] == category]
    groups=[s for s in timing['streams'] if s['path'].split('/')[0] == category]
    rows.append([category, len(files), sum(f['records'] for f in files),
                 sum(f['finite_timestamps'] for f in files), len(groups)])
show_table(['Folder category', 'Files', 'Source records', 'Finite timestamps', 'Diagnostic identity groups'], rows)
print('All source records:', sum(f['records'] for f in timing['source_files']))
print('Records in diagnostic identity groups:', sum(s['records'] for s in timing['streams']))
print('Records without usable identity:', len(timing['unusable_stream_identity_records']))
print('Diagnostic identity groups:', len(timing['streams']))

Folder category,Files,Source records,Finite timestamps,Diagnostic identity groups
Benign,12,11598,10664,14
BruteForce,4,3057,3057,4
DDOS,9,15084,15084,43
DoS,10,8187,3929,6
Probe,7,9014,8445,13
Web,7,3273,3273,8


All source records: 50213
Records in diagnostic identity groups: 44452
Records without usable identity: 5761
Diagnostic identity groups: 88


### 4. Examine original-order deltas in raw units

The diagnostic preserves within-file stream order. It does not sort timestamps
before looking for reversals. Missing/invalid timestamps break adjacency;
identified observations need not be consecutive physical reports if other
rows cannot be assigned to a stream. Median/max positive steps are descriptive
only. No gap threshold has been defined, and no gap is assigned a cause.

A median step of 1,000 raw units is consistent with one-second sampling **if**
the timestamps are milliseconds. Neither the magnitude nor the histogram alone
validates epoch, units, event/arrival semantics or synchronization. No seconds
or dates are inferred here.

In [18]:
print('Streams with backward steps:', sum(s['negative_steps'] > 0 for s in timing['streams']))
print('Streams with zero steps:', sum(s['zero_steps'] > 0 for s in timing['streams']))
print('Repeated timestamp occurrences within streams:', sum(s['repeated_timestamp_occurrences'] for s in timing['streams']))
median_counts=Counter(s['positive_step_median_raw'] for s in timing['streams'])
show_table(['Median positive step (raw units)', 'Identity groups'], sorted(median_counts.items(), key=lambda p: str(p[0])))
largest=sorted(timing['streams'], key=lambda s: s['positive_step_max_raw'] or 0, reverse=True)[:12]
show_table(['Stream index (not run ID)', 'Source path', 'Records', 'Median positive step (raw)',
            'Largest positive step (raw)', 'Backward steps', 'Zero steps'], [
    [s['stream_index'], s['path'], s['records'], s['positive_step_median_raw'],
     s['positive_step_max_raw'], s['negative_steps'], s['zero_steps']]
    for s in largest
])

Streams with backward steps: 0
Streams with zero steps: 0
Repeated timestamp occurrences within streams: 0


Median positive step (raw units),Identity groups
1000,88


Stream index (not run ID),Source path,Records,Median positive step (raw),Largest positive step (raw),Backward steps,Zero steps
0,Benign/Benign/Lower_Layer/benign.txt,954,1000,143993,0,0
1,Benign/Benign/Lower_Layer/benign.txt,954,1000,143992,0,0
2,Benign/Benign/Lower_Layer/benign.txt,449,1000.0,143992,0,0
65,DoS/DoS/Lower_Layer/tcp_npig.txt,748,1000,125781,0,0
12,Benign/Benign/Lower_Layer/benign_8.txt,1520,1000,105486,0,0
67,Probe/Probe/Lower_Layer/osfingerprinting_nmap.txt,3273,1000.0,35076,0,0
72,Probe/Probe/Lower_Layer/portscan_tcp1.txt,330,1000,23133,0,0
16,BruteForce/BruteForce/Lower_Layer/ssh_hydra.txt,924,1000,22089,0,0
68,Probe/Probe/Lower_Layer/osfingerprinting_nmap.txt,2374,1000,16626,0,0
20,DDOS/DDOS/Lower_Layer/ddos_ack_nping.txt,360,1000,1070,0,0


### 5. Distinguish range overlap from shared observations

Only cross-file pairs with the exact same cell/UE/RNTI identity and at least
one usable clock value on both sides are compared. Ranges overlap inclusively
when their min/max intervals intersect. The report separately counts common
timestamps and shared exact radio-plus-metadata occurrences, preserving
multiplicity. A range overlap alone is not a repeated observation.

No overlap under this key does **not** prove independent captures: identities
can rotate, unseen data may exist, and clocks themselves need validation.

In [19]:
overlaps=timing['overlap_diagnostics']
print('Same-identity cross-file pairs compared:', overlaps['same_identity_cross_file_pairs_compared'])
print('Pairs with overlapping raw timestamp ranges:', len(overlaps['range_overlaps']))
if overlaps['range_overlaps']:
    show_table(['Left file', 'Right file', 'Common distinct timestamps', 'Shared radio/metadata occurrences'], [
        [o['left_path'], o['right_path'], o['common_distinct_timestamps'], o['shared_radio_metadata_occurrences']]
        for o in overlaps['range_overlaps']
    ])
else:
    print('No range overlaps found within this exact identity comparison; independence is unproven.')

Same-identity cross-file pairs compared: 102
Pairs with overlapping raw timestamp ranges: 0
No range overlaps found within this exact identity comparison; independence is unproven.


### 6. Investigate populated metadata not observed in candidate source records

Notebook 02 identified unique radio candidates whose sources cannot supply
the stronger metadata key. For those database rows, compare the actual
`timestamp`, `ue_id`, `cellid` and `in_sync` against all source observations.
RNTI is not used for this separate coincidence test, and missing fields are
never wildcards. This is explicitly weaker than a measurement identity.

Every coincident source path/line is retained, along with a flag recording
whether the 16 radio values also agree. Timestamp/identity coincidence with
different radio values raises an association question; it does not establish
copying, synthetic clocks, source corruption or the intended processing rule.

In [20]:
cases=timing['questionable_metadata_coincidences']
print('Questionable database rows examined:', len(cases))
show_table(['Coincident source observations per database row', 'Database rows'],
           sorted(Counter(len(c['coincident_source_observations']) for c in cases).items()))
hits=Counter()
for case in cases:
    for hit in case['coincident_source_observations']:
        hits[(case['source_path'], hit['path'])] += 1
show_table(['Metadata-free radio candidate source', 'Other source with coincident metadata', 'Database rows'],
           [[*key,count] for key,count in sorted(hits.items())])
print('Coincident observations also sharing the 16 radio values:',
      sum(hit['radio_values_also_equal'] for c in cases for hit in c['coincident_source_observations']))
print('Unusable database coincidence keys:', sum(c['comparison_issue'] is not None for c in cases))

Questionable database rows examined: 1540


Coincident source observations per database row,Database rows
1,1540


Metadata-free radio candidate source,Other source with coincident metadata,Database rows
DoS/DoS/Lower_Layer/tcp-ack_hping3.txt,DoS/DoS/Lower_Layer/icmp_nping.txt,199
DoS/DoS/Lower_Layer/tcp-ack_hping3.txt,DoS/DoS/Lower_Layer/slowloris2.txt,213
DoS/DoS/Lower_Layer/tcp-ack_hping3.txt,DoS/DoS/Lower_Layer/udp_nping.txt,236
DoS/DoS/Lower_Layer/tcp-syn_hping3.txt,DoS/DoS/Lower_Layer/icmp_nping.txt,243
DoS/DoS/Lower_Layer/tcp-syn_hping3.txt,DoS/DoS/Lower_Layer/slowloris2.txt,272
DoS/DoS/Lower_Layer/tcp-syn_hping3.txt,DoS/DoS/Lower_Layer/udp_nping.txt,253
DoS/DoS/Lower_Layer/tcp_test.txt,DoS/DoS/Lower_Layer/icmp_nping.txt,45
DoS/DoS/Lower_Layer/tcp_test.txt,DoS/DoS/Lower_Layer/slowloris2.txt,44
DoS/DoS/Lower_Layer/tcp_test.txt,DoS/DoS/Lower_Layer/udp_nping.txt,35


Coincident observations also sharing the 16 radio values: 0
Unusable database coincidence keys: 0


### 7. Inspect available provenance without executing external code

The archive search covers filenames/extensions only; it does not establish
that metadata are absent from every JSON/CSV field. Packaging timestamps are
not recording timestamps.

Separately, the authors' repository was reviewed read-only at commit
`add5de9fe2a1491a8d0bf4ccf0239c553c286634`. Its radio notebook uses legacy
Server_Data/database names; the two inspected notebooks do not describe the
current `Lower_Layer_Data.db` construction or resolve the questionable DoS
metadata. This is a limit of the inspected snapshot, not proof that no
processing records exist elsewhere. See `docs/provenance_sources.md` for
pinned source links and review scope. No external notebooks were executed.

In [21]:
archive=timing['archive_inventory']
print('Archive inspection:', archive['status'])
if 'extension_counts' in archive:
    print('Archive bytes:', archive['bytes'], '; file entries:', archive['file_entries'])
    show_table(['Extension', 'File entries'], sorted(archive['extension_counts'].items()))
    print('Provenance filename candidates:', archive['provenance_filename_candidates'])
    print('Search limitation:', archive['limitation'])

Archive inspection: central directory inspected only
Archive bytes: 25101375790 ; file entries: 320


Extension,File entries
.csv,110
.db,4
.json,39
.pcap,83
.pcapng,35
.txt,49


Provenance filename candidates: []
Search limitation: Filename/extension search only; JSON/CSV content not searched, no full archive integrity/hash check. ZIP packaging dates are not capture dates.


### 8. Decision gate and next task

The diagnostic results can describe source clocks, but they cannot convert
identity groups into independent runs. Obtain author/team capture notes or
export scripts and the exact Kaggle revision. Investigate how populated
metadata became associated with the 1,540 DoS radio candidates. Until that is
resolved, neither replacing those clocks nor discarding the rows is approved.

The next evaluation proposal should explicitly separate documented evidence
from fallback assumptions. Gaurav must review any population restriction,
run/group rule, time interpretation or teammate interface change. Pending
provenance does not authorize tuning a split to improve model performance.

No date/class separability, benign independence, counter rates or held-out
performance is established here. The preliminary 52 versus 42 runs remains
unverified. Nitendra owns models/calibration; Marco owns degradation/playback.

In [22]:
assert timing['input_hashes_unchanged']
assert sum(s['records'] for s in timing['streams']) + len(timing['unusable_stream_identity_records']) == sum(f['records'] for f in timing['source_files'])
print('All source records accounted for; extracted input hashes unchanged.')
print('No raw-unit conversion, gap threshold, independent-run assignment, row exclusion or model training performed.')

All source records accounted for; extracted input hashes unchanged.
No raw-unit conversion, gap threshold, independent-run assignment, row exclusion or model training performed.


### 9. Follow-up: can database clocks preserve candidate source order?

Gaurav confirmed that no Kaggle revision identifier or team export notes are
available. The snapshot is therefore identified by the recorded file hashes;
no platform revision is invented. We cannot wait for unavailable internal
processing records, but we must retain uncertainty about the original mechanism.

Project the questionable database timestamps onto the physical line order of
their unique radio-candidate source records. These are **database clocks**, not
observed source timestamps. Count reused metadata tuples globally and within
files, without removing repeats. Physical source order itself is not a
certified event-time order, so reversals are a warning rather than proof of
fabrication, a particular resampling method, or attack timing.

In [23]:
from reliability.metadata_provenance import summarize_projected_clocks
projected = summarize_projected_clocks(timing['questionable_metadata_coincidences'])
print('Projected-clock helper SHA-256:', hashlib.sha256((PROJECT_ROOT / 'src/reliability/metadata_provenance.py').read_bytes()).hexdigest())
print('Questionable database rows:', projected['cases'])
print('Distinct usable metadata tuples:', projected['distinct_usable_metadata_tuples'])
print('Repeated tuple occurrences beyond one each:', projected['repeated_tuple_excess'])
show_table(['Radio candidate source', 'Rows', 'Distinct metadata tuples', 'Repeat excess',
            'Backward projected steps', 'Forward projected steps', 'Zero projected steps'], [
    [f['path'], f['cases'], f['distinct_usable_metadata_tuples'], f['repeated_tuple_excess'],
     f['projected_database_clock_in_source_line_order']['negative_steps'],
     f['projected_database_clock_in_source_line_order']['positive_steps'],
     f['projected_database_clock_in_source_line_order']['zero_steps']]
    for f in projected['files']
])
print('Interpretation:', projected['interpretation'])

Projected-clock helper SHA-256: 89c857fbbd67b31b8cd79b0a023d41e4ccd81d9c0941428bfe578080cd62f83d
Questionable database rows: 1540
Distinct usable metadata tuples: 1143
Repeated tuple occurrences beyond one each: 397


Radio candidate source,Rows,Distinct metadata tuples,Repeat excess,Backward projected steps,Forward projected steps,Zero projected steps
DoS/DoS/Lower_Layer/tcp-ack_hping3.txt,648,572,76,317,330,0
DoS/DoS/Lower_Layer/tcp-syn_hping3.txt,768,655,113,383,383,1
DoS/DoS/Lower_Layer/tcp_test.txt,124,121,3,60,63,0


Interpretation: Database clock projection onto candidate source line order; not measured source event time or proof of a metadata assignment algorithm.


#### Working conclusion and research boundary

The observable question is whether timing is supported by the available source
records. The exact author assignment procedure may be unrecoverable from final
files. Record both the positive evidence and that limit. Do not label these
clocks synthetic or repaired as an established fact.

Any proposal to exclude rows from temporal evaluation, analyze them separately,
interpret timestamps as milliseconds, or use source-file groups must be
presented for Gaurav's decision before implementation. No such choice is made
in this notebook. File hashes identify the available content, not a missing
platform revision or independent captures.